A loss turns predictions and observations into an objective for parameter
search. Fitters consume the `LossFn` interface, so the same fitting procedure
can use sequence likelihood or a discrepancy between behavioral summaries.

## The loss interface

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class LossFn(Protocol):
    def __init__(
        self,
        model_factory_cls: type[MemorySearchModelFactory],
        dataset: RecallDataset,
        features: Float[Array, " word_pool_items features_count"] | None,
    ) -> None: ...

    def __call__(
        self,
        trial_indices: Integer[Array, " trials"],
        base_params: Mapping[str, Float_],
        free_param_names: Iterable[str],
        x: Float[Array, " parameters n_samples"],
    ) -> Float[Array, " n_samples"]: ...
```

The constructor binds the dataset, optional features and model factory.
A compatible **loss class** is supplied to a fitter as `loss_fn_cls`;
the fitter constructs its callable instance.

| Call argument | Responsibility |
|---|---|
| `trial_indices` | Explicit dataset row indices to score, rather than a Boolean mask. |
| `base_params` | Fixed values and defaults. Free values override matching entries. |
| `free_param_names` | Names in the order of the candidate matrix's parameter axis. |
| `x` | Physical model parameters shaped `(parameters, candidates)`. Each column is one candidate. |

The result has one objective per candidate. Fitters minimize these values.
A custom loss must preserve this argument order, parameter mapping and
output shape and support the JAX operations used by its chosen fitter.
Search-coordinate transformations belong to the fitting adapter; the loss
receives model parameter values.

## Likelihood and event masks

Sequence likelihood follows the observed response history, obtains the
probability of each event and sums its negative log probability. A
`LikelihoodMaskFn` chooses which events contribute to that sum.

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class LikelihoodMaskFn(Protocol):
    def __call__(
        self, recalls: Integer[Array, " recall_events"]
    ) -> Bool[Array, " recall_events"]: ...
```

In `jaxcmr.loss.transform_sequence_likelihood`, `MemorySearchLikelihoodLoss`
accepts this callback as `mask_likelihoods`. It translates recorded study
positions to within-list item codes before calling the mask. The existing
`mask_trailing_terminations()` returns `recalls != 0`.

The convenient `ExcludeTerminationLikelihoodLoss` binds that mask and
provides the standard three-argument `LossFn` constructor used by fitters.
With the no-stop model used here, the objective scores the recorded item
sequence without scoring its length. Zero-coded slots do not contribute.
An excluded event does not erase the
preceding history that conditions later predictions. A first-recall mask,
for example, removes that event's score while later events still follow it.

In [1]:
#| code-summary: "Evaluate two parameter columns through the loss interface"

from pathlib import Path
import json
import jax
import jax.numpy as jnp
from jaxcmr.helpers import load_data, generate_trial_mask
from jaxcmr.components import context, linear_memory
from jaxcmr.components.termination import NoStopTermination
from jaxcmr.models.cmr import make_factory

jax.config.update("jax_enable_x64", True)
data = load_data("data/Dupertuys2026.h5")
record = json.loads(Path("data/fits/Dupertuys2026_CMR_demo.json").read_text())
parameters = {name: values[0] for name, values in record["fits"].items()
              if name != "subject"}
for name in ("allow_repeated_recalls", "learn_after_context_update"):
    parameters[name] = bool(parameters[name])

CMRFactory = make_factory(
    mfc_create_fn=linear_memory.init_mfc,
    mcf_create_fn=linear_memory.init_mcf,
    context_create_fn=context.init,
    termination_policy_create_fn=NoStopTermination,
)

from jaxcmr.loss.transform_sequence_likelihood import ExcludeTerminationLikelihoodLoss

mixed = generate_trial_mask(data, "data['list_type'] == 1")
trial_indices = jnp.flatnonzero(mixed)[:1]
free_names = ("encoding_drift_rate", "recall_drift_rate")
base_params = {name: value for name, value in parameters.items() if name not in free_names}
candidates = jnp.array([
    [parameters["encoding_drift_rate"], 0.2],
    [parameters["recall_drift_rate"], parameters["recall_drift_rate"]],
])

loss = ExcludeTerminationLikelihoodLoss(CMRFactory, data, features=None)
values = loss(trial_indices, base_params, free_names, candidates)
print("Parameter order:", free_names)
print("Candidate matrix shape:", candidates.shape)
print("Objective shape:", values.shape)
print("Negative log likelihoods:", values)

Parameter order: ('encoding_drift_rate', 'recall_drift_rate')
Candidate matrix shape: (2, 2)
Objective shape: (2,)
Negative log likelihoods: [13.62458774 13.05021919]


Only one mixed list is scored here to expose the interface. The first column
uses the supplied fit's parameter values; the second changes encoding drift.
The returned values are not the objectives from fitting the whole experiment.

The underlying likelihood also exposes event probabilities. Setting masked
probabilities to one makes their contribution `-log(1) = 0`.

In [2]:
#| code-summary: "Connect the event probabilities and mask to the returned NLL"

from jaxcmr.helpers import log_likelihood
from jaxcmr.loss.transform_sequence_likelihood import (
    MemorySearchLikelihoodLoss, mask_trailing_terminations,
)

likelihood = MemorySearchLikelihoodLoss(
    CMRFactory, data, features=None,
    mask_likelihoods=mask_trailing_terminations,
)
row = trial_indices[0]
probabilities = likelihood.predict_trial(row, parameters)
include = mask_trailing_terminations(data["recalls"][row])
event_nll = log_likelihood(jnp.where(include, probabilities, 1.0))
print("Included event probabilities:", probabilities[include])
print("Included events:", int(include.sum()))
print("NLL from event probabilities:", float(event_nll))
print("NLL from the loss call:", float(values[0]))

Included event probabilities: [0.13703638 0.04187474 0.08562368 0.19103804 0.0785633  0.16413277]
Included events: 6
NLL from event probabilities: 13.624587741452169
NLL from the loss call: 13.62458774145214


Despite its name, `jaxcmr.helpers.log_likelihood()` returns **negative summed
log likelihood**. The mask selects the positive recall events; it does not
define a new response sequence.

## Behavioral-summary objectives

`jaxcmr.loss.mse.make_factory()` binds named `RecallAnalysisFn` callbacks into
a loss class with the same fitting interface. It compares observed and
simulated summaries rather than assigning a likelihood to each recorded
response. [Recall Analyses](analyses.ipynb#recallanalysisfn) shows the callback
contract and an adapter using the existing SPC calculation.

Its weights, scales and reduction determine how summary discrepancies
contribute to the objective. Such a value is not NLL and cannot be put into
an AIC formula. JAX-compatible simulation also does not by itself provide
useful pathwise gradients through discrete recall samples.

## Preparing a fixed selection

`PreparableLossFn` is an optional capability used by fitting adapters to
prepare observations before JIT compilation. An ordinary loss need only
implement `LossFn`.

Declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class PreparableLossFn(LossFn, Protocol):
    def prepare_trials(
        self, trial_indices: Integer[Array, " trials"]
    ) -> PreparedLossFn: ...
```

The returned `PreparedLossFn` takes `(base_params, free_param_names, x)` and
returns the same candidate-wise objectives. The selected trials are already
bound, including their order and multiplicity. Each prepared callable must
retain its own selection. `BatchPreparableLossFn` extends this to several
selections so implementations can share compatible preparation work.

In [3]:
#| code-summary: "Evaluate the same candidates with the selected trials already bound"

prepared = loss.prepare_trials(trial_indices)
prepared(base_params, free_names, candidates)

Array([13.62458774, 13.05021919], dtype=float64)

The prepared call represents the same objective. Preparation is a capability
of the implementation, not a change in the scientific quantity being fitted.
[Fitting Procedures](fitting.ipynb) explains how fitters consume these losses;
the [fitting template](../templates/fitting_scipyde.ipynb) carries out a full run.